# BERT + A\* vs LLMs — Word Ladder comparison

Compare your fine-tuned distance-regression model (A\* + BFS fallback) against **OpenAI Chat Completions** and (optionally) **Google Gemini** on the **same** random word pairs.

**“Not in vocab”** means the LLM proposed a word that is **not a node in your island file** — we never send the full lexicon to the API (too many tokens). The model guesses common words; they often are not in *your* graph.

- **`GPT_REQUIRE_GRAPH_VOCAB`**: applies to **both** GPT and Gemini eval loops — `True` = every step must be a valid edge in your graph (fair vs your BERT solver); `False` = only length + one-letter steps + start/end match (“relaxed” ladder).

The chat prompt is built in **`build_gpt_chat_payload`** (helpers cell); **GPT** and **Gemini** reuse the same system + user text. Each API evaluation cell prints a copy for the first eval pair.

**Configure everything in the next code cell** (`GRAPH_PRESET`, `GPT_MODEL`, `GEMINI_MODEL`, etc.).

Eval pairs: with **`EVAL_SET_NAME = None`**, the notebook picks **`data/eval_sets/<name>.csv`** from **`GRAPH_PRESET`** (e.g. `english_4` → `test4english.csv`). If that file is missing, it **samples** `N_PAIRS`, saves the CSV, and reuses it on later runs.

**Requirements:**
- Trained model in `models/bert_wordladder_<preset>/` (download from Colab if needed)
- Island files under `data/islands/`
- **`OPENAI_API_KEY`** in **`.env.local`** (or config cell) for GPT runs
- **`GEMINI_API_KEY`** in **`.env.local`** for Gemini — see `.env.local.example`; **do not commit secrets**
- `pip install openai python-dotenv google-genai transformers torch pandas tqdm networkx`

In [83]:
# =============================================================================
# CONFIG — edit this cell only (then run all below in order)
# =============================================================================

# Graph + your BERT checkpoint (must match training)
# One of: "english_5" | "english_4" | "croatian_5" | "croatian_4"
GRAPH_PRESET = "croatian_4"

# OpenAI *model id* (exact API string, no spaces). Common:
#   gpt-4o-mini  — cheap / fast (default)
#   gpt-4o       — stronger, higher cost
GPT_MODEL = "gpt-5.4-mini"

# API key: None → os.environ (after loading repo-root `.env.local` in the imports cell)
OPENAI_API_KEY = None

# Eval set: data/eval_sets/<EVAL_SET_NAME>.csv
# None or "" → name is picked from GRAPH_PRESET (test5english, test4english, test5croatian, test4croatian).
# Set a string only to override; it must match graph_preset inside that CSV.
EVAL_SET_NAME = None
# If the file exists, load it (same word pairs every run). If missing, sample and save.
EVAL_SET_REUSE_IF_EXISTS = True
# True = always resample N_PAIRS and overwrite the eval-set file (ignores reuse)
FORCE_RESAMPLE_EVAL = False

N_PAIRS = 20  # used only when creating a new eval set file
RANDOM_SEED = 42
DIST_MIN, DIST_MAX = 3, 10

MAX_EXPANSIONS = 300
MAX_STEPS = 25

GPT_TEMPERATURE = 0.2
GPT_MAX_TOKENS = 512
# gpt-5.x and some models reject max_tokens — use max_completion_tokens instead
GPT_USE_MAX_COMPLETION_TOKENS = True
# If you get 400 on max_completion_tokens with gpt-4o-mini, set False

# If True, GPT paths must use only words from the island graph (same as your game).
# If False, only length + one-letter steps are checked (no full-vocab prompt; path may be off-graph).
GPT_REQUIRE_GRAPH_VOCAB = False

# Optional: explicit CSV paths (override EVAL_SET_NAME if set — highest priority for loading)
LOAD_PAIRS_CSV = None  # e.g. Path or str to a custom file
SAVE_PAIRS_CSV = None  # if set on *sample* path, save here instead of eval_sets/<name>.csv

# Google Gemini — same prompt as GPT (`build_gpt_chat_payload`). Needs `pip install google-genai`.
# Set GEMINI_MODEL = "" to skip the Gemini benchmark cell (no API calls).
GEMINI_MODEL = "gemini-2.5-flash"
GEMINI_API_KEY = None  # None → os.environ["GEMINI_API_KEY"] after dotenv
GEMINI_TEMPERATURE = 0.2
GEMINI_MAX_OUTPUT_TOKENS = 512

REPO_ROOT = None  # None = auto from notebooks/ folder


In [49]:
# Optional: install deps (run once per environment)
%pip install openai python-dotenv google-genai -q

Note: you may need to restart the kernel to use updated packages.


In [84]:
from __future__ import annotations

import heapq
import json
import os
import random
import re
import string
import time
from collections import deque
from pathlib import Path

import networkx as nx
import pandas as pd
import torch
from openai import OpenAI
from transformers import AutoModelForSequenceClassification, AutoTokenizer

try:
    from tqdm import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

if REPO_ROOT is None:
    REPO_ROOT = Path("..").resolve()
else:
    REPO_ROOT = Path(REPO_ROOT).resolve()

try:
    from dotenv import load_dotenv

    load_dotenv(REPO_ROOT / ".env.local")  # your real key (gitignored)
    load_dotenv(REPO_ROOT / ".env")       # optional fallback
except ImportError:
    pass

DATA = REPO_ROOT / "data"
ISLANDS = DATA / "islands"
MODELS = REPO_ROOT / "models"

CROATIAN_LETTERS = "abcdefghijklmnopqrstuvwxyz\u010d\u0107\u0111\u0161\u017e"

PRESETS = {
    "english_5": {
        "model_subdir": "bert_wordladder_5letter",
        "largest": ISLANDS / "english_5_largest_island.txt",
        "strict": ISLANDS / "english_5_strict_largest_island.txt",
        "letters": string.ascii_lowercase,
        "lang_note": "English",
    },
    "english_4": {
        "model_subdir": "bert_wordladder_4letter",
        "largest": ISLANDS / "english_4_largest_island.txt",
        "strict": ISLANDS / "english_4_strict_largest_island.txt",
        "letters": string.ascii_lowercase,
        "lang_note": "English",
    },
    "croatian_5": {
        "model_subdir": "bert_wordladder_croatian5",
        "largest": ISLANDS / "croatian_5_largest_island.txt",
        "strict": ISLANDS / "croatian_5_strict_largest_island.txt",
        "letters": CROATIAN_LETTERS,
        "lang_note": "Croatian (letters a-z plus \u010d, \u0107, \u0111, \u0161, \u017e; words are stored as Unicode strings)",
    },
    "croatian_4": {
        "model_subdir": "bert_wordladder_croatian4",
        "largest": ISLANDS / "croatian_4_largest_island.txt",
        "strict": ISLANDS / "croatian_4_strict_largest_island.txt",
        "letters": CROATIAN_LETTERS,
        "lang_note": "Croatian (letters a-z plus \u010d, \u0107, \u0111, \u0161, \u017e)",
    },
}

if GRAPH_PRESET not in PRESETS:
    raise ValueError(f"GRAPH_PRESET must be one of {list(PRESETS)}")

cfg = PRESETS[GRAPH_PRESET]
MODEL_PATH = MODELS / cfg["model_subdir"]
FULL_ISLAND = cfg["largest"]
STRICT_ISLAND = cfg["strict"]
LETTERS = cfg["letters"]
LANG_NOTE = cfg["lang_note"]
MAX_LENGTH = 32

_sample = next((w for w in FULL_ISLAND.read_text(encoding="utf-8").splitlines() if w.strip()), "")
WORD_LEN = len(_sample.strip())
print(f"Preset: {GRAPH_PRESET} | word length: {WORD_LEN} | model: {MODEL_PATH.name}")
print(f"Island: {FULL_ISLAND.name}, strict: {STRICT_ISLAND.name}")

_EVAL_SET_BY_PRESET = {
    "english_5": "test5english",
    "english_4": "test4english",
    "croatian_5": "test5croatian",
    "croatian_4": "test4croatian",
}
_eval_raw = globals().get("EVAL_SET_NAME", None)
if _eval_raw is None or (isinstance(_eval_raw, str) and not str(_eval_raw).strip()):
    EVAL_SET_NAME = _EVAL_SET_BY_PRESET[GRAPH_PRESET]
    print(f"Eval set: {EVAL_SET_NAME} (auto for {GRAPH_PRESET})")
else:
    EVAL_SET_NAME = str(_eval_raw).strip()
    print(f"Eval set: {EVAL_SET_NAME} (manual override)")

GPT_MODEL = str(GPT_MODEL).strip()
if not GPT_MODEL:
    raise ValueError("GPT_MODEL is empty — use e.g. gpt-4o-mini (see config cell).")

# Default True if config cell predates this flag (avoids NameError on partial re-run)
GPT_USE_MAX_COMPLETION_TOKENS = bool(globals().get("GPT_USE_MAX_COMPLETION_TOKENS", True))
GPT_REQUIRE_GRAPH_VOCAB = bool(globals().get("GPT_REQUIRE_GRAPH_VOCAB", True))

api_key = OPENAI_API_KEY or os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENAI_API_KEY in the config cell or in the environment.")
openai_client = OpenAI(api_key=api_key)
print(f"GPT model: {GPT_MODEL}")

GEMINI_MODEL = str(globals().get("GEMINI_MODEL", "") or "").strip()
_g_key = GEMINI_API_KEY or os.environ.get("GEMINI_API_KEY")
gemini_client = None
if GEMINI_MODEL and _g_key:
    try:
        from google import genai
    except ImportError as e:
        raise ImportError("pip install google-genai (see pip cell) to use Gemini.") from e
    gemini_client = genai.Client(api_key=_g_key)
    print(f"Gemini model: {GEMINI_MODEL}")
elif GEMINI_MODEL and not _g_key:
    print("Warning: GEMINI_MODEL is set but GEMINI_API_KEY is missing — Gemini cell will error if run.")
else:
    print("Gemini: skipped (set GEMINI_MODEL to enable; key from .env.local)")


Preset: croatian_4 | word length: 4 | model: bert_wordladder_croatian4
Island: croatian_4_largest_island.txt, strict: croatian_4_strict_largest_island.txt
Eval set: test4croatian (auto for croatian_4)
GPT model: gpt-5.4-mini
Gemini model: gemini-2.5-flash


In [85]:
def load_vocab(path: Path) -> set[str]:
    return {w.strip().lower() for w in path.read_text(encoding="utf-8").splitlines() if w.strip()}


def one_letter_neighbors(w: str, vocab: set[str]) -> set[str]:
    nbs = set()
    for i in range(len(w)):
        for c in LETTERS:
            if c != w[i]:
                cand = w[:i] + c + w[i + 1 :]
                if cand in vocab:
                    nbs.add(cand)
    return nbs


def is_one_step(a: str, b: str) -> bool:
    if len(a) != len(b):
        return False
    return sum(1 for x, y in zip(a, b) if x != y) == 1


def validate_ladder(
    path: list[str],
    vocab: set[str],
    word_len: int,
    require_in_graph_vocab: bool = True,
) -> tuple[bool, str]:
    """Return (ok, reason). If require_in_graph_vocab is False, skip membership in `vocab`."""
    if not path:
        return False, "empty path"
    for w in path:
        if len(w) != word_len:
            return False, f"bad length: {w!r}"
        if require_in_graph_vocab and w not in vocab:
            return False, f"not in graph vocab: {w!r}"
    for i in range(len(path) - 1):
        if not is_one_step(path[i], path[i + 1]):
            return False, f"not one-letter step: {path[i]!r} -> {path[i+1]!r}"
    return True, ""


def fraction_words_in_graph(path: list[str] | None, vocab: set[str]) -> float:
    if not path:
        return 0.0
    return sum(1 for w in path if w in vocab) / len(path)


def score_candidates(model, tokenizer, current: str, target: str, candidates: list, device):
    if not candidates:
        return []
    enc = tokenizer(
        list(candidates),
        [target] * len(candidates),
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    model.eval()
    with torch.no_grad():
        preds = model(**enc).logits.squeeze(-1).cpu().tolist()
    if isinstance(preds, float):
        preds = [preds]
    return sorted(zip(candidates, preds), key=lambda x: x[1])


def shortest_path_bfs(start: str, target: str, vocab: set[str]) -> list[str] | None:
    if start not in vocab or target not in vocab:
        return None
    q = deque([(start, [start])])
    seen = {start}
    while q:
        w, path = q.popleft()
        if w == target:
            return path
        for n in one_letter_neighbors(w, vocab) - seen:
            seen.add(n)
            q.append((n, path + [n]))
    return None


def generate_path_astar(
    model, tokenizer, vocab: set[str], start: str, target: str,
    device, max_expansions: int = 300, max_steps: int = 25,
) -> tuple[list[str] | None, str | None]:
    start, target = start.lower().strip(), target.lower().strip()
    if start not in vocab or target not in vocab:
        return None, None
    h0 = max(score_candidates(model, tokenizer, start, target, [start], device)[0][1], 0.0)
    counter = 0
    pq = [(h0, counter, [start])]
    best_g = {start: 0}
    expansions = 0
    while pq and expansions < max_expansions:
        _, _, path = heapq.heappop(pq)
        current = path[-1]
        g = len(path) - 1
        if current == target:
            return path, "astar"
        if g >= max_steps:
            continue
        if g > best_g.get(current, float("inf")):
            continue
        neighbors = list(one_letter_neighbors(current, vocab))
        if not neighbors:
            continue
        ranked = score_candidates(model, tokenizer, current, target, neighbors, device)
        expansions += 1
        for word, pred in ranked:
            g2 = g + 1
            if g2 < best_g.get(word, float("inf")):
                best_g[word] = g2
                h = max(pred, 0.0)
                counter += 1
                heapq.heappush(pq, (g2 + h, counter, path + [word]))
    bfs = shortest_path_bfs(start, target, vocab)
    return (bfs, "bfs") if bfs else (None, None)


def parse_gpt_path(raw: str, word_len: int) -> list[str] | None:
    """Extract JSON array of strings from model output."""
    raw = raw.strip()
    # strip fenced code blocks
    m = re.search(r"\[[\s\S]*\]", raw)
    if not m:
        return None
    try:
        data = json.loads(m.group(0))
    except json.JSONDecodeError:
        return None
    if not isinstance(data, list):
        return None
    out = []
    for x in data:
        if not isinstance(x, str):
            return None
        w = x.strip().lower()
        if len(w) != word_len:
            return None
        out.append(w)
    return out


def build_gpt_chat_payload(start: str, target: str, word_len: int) -> tuple[str, str, list]:
    """System text, user text, and OpenAI messages list (same as sent to the API)."""
    start, target = start.lower().strip(), target.lower().strip()
    sys = (
        "You solve word ladders. Reply with ONLY a JSON array of lowercase strings: "
        "one word per step from start to target, no markdown, no commentary outside the array."
    )
    if word_len == 4:
        worked = (
            'Tiny example (4-letter, lowercase). First step in full: heat→head — change position 4 from t to d. '
            'Then only the deltas: head→heal (4:d→l); heal→hell (3:a→l). '
            'JSON: ["heat","head","heal","hell"]'
        )
    else:
        worked = (
            'Tiny example (5-letter, lowercase). First step in full: fears→bears — change position 1 from f to b. '
            'Then only the deltas: bears→beads (4:r→d); beads→reads (1:b→r); reads→ready (5:s→y). '
            'JSON: ["fears","bears","beads","reads","ready"]'
        )
    user = (
        f"Language: {LANG_NOTE}.\n"
        f"Each word must have exactly {word_len} characters.\n"
        f"Each consecutive pair must differ in exactly ONE character position (same length).\n\n"
        f"{worked}\n\n"
        f"--- Your task ---\n"
        f"Start: \"{start}\"\n"
        f"Target: \"{target}\"\n\n"
        f"Output ONLY a JSON array from start to target, e.g. [\"{start}\", \"...\", \"{target}\"]"
    )
    msg = [{"role": "system", "content": sys}, {"role": "user", "content": user}]
    return sys, user, msg


def gpt_word_ladder(start: str, target: str, word_len: int) -> tuple[list[str] | None, str, float]:
    """Call OpenAI; return (path_or_none, raw_text, latency_sec)."""
    _, _, msg = build_gpt_chat_payload(start, target, word_len)
    t0 = time.time()
    use_mct = globals().get("GPT_USE_MAX_COMPLETION_TOKENS", True)
    params = dict(model=GPT_MODEL, messages=msg, temperature=GPT_TEMPERATURE)
    if use_mct:
        params["max_completion_tokens"] = GPT_MAX_TOKENS
    else:
        params["max_tokens"] = GPT_MAX_TOKENS
    resp = openai_client.chat.completions.create(**params)
    elapsed = time.time() - t0
    raw = resp.choices[0].message.content or ""
    path = parse_gpt_path(raw, word_len)
    return path, raw, elapsed


def gemini_word_ladder(start: str, target: str, word_len: int) -> tuple[list[str] | None, str, float]:
    """Call Gemini (google-genai); same prompt as GPT. Returns (path_or_none, raw_text, latency_sec)."""
    if gemini_client is None:
        raise RuntimeError("Gemini client not configured — set GEMINI_API_KEY and GEMINI_MODEL, run imports cell.")
    from google.genai import types

    sys_txt, user_txt, _ = build_gpt_chat_payload(start, target, word_len)
    t0 = time.time()
    resp = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=user_txt,
        config=types.GenerateContentConfig(
            system_instruction=sys_txt,
            temperature=float(globals().get("GEMINI_TEMPERATURE", 0.2)),
            max_output_tokens=int(globals().get("GEMINI_MAX_OUTPUT_TOKENS", 512)),
        ),
    )
    elapsed = time.time() - t0
    raw = ""
    try:
        raw = (getattr(resp, "text", None) or "").strip()
    except Exception:
        raw = ""
    path = parse_gpt_path(raw, word_len)
    return path, raw, elapsed

print("Helpers OK")

Helpers OK


In [86]:
vocab = load_vocab(FULL_ISLAND)
playable = sorted(load_vocab(STRICT_ISLAND))
if not playable:
    raise RuntimeError(f"No words in {STRICT_ISLAND}")

G = nx.Graph()
G.add_nodes_from(vocab)
for w in tqdm(vocab, desc="Graph edges"):
    for nb in one_letter_neighbors(w, vocab):
        if w < nb:
            G.add_edge(w, nb)

EVAL_SETS_DIR = DATA / "eval_sets"
_name = (str(EVAL_SET_NAME).strip() if EVAL_SET_NAME else "")
EVAL_SET_PATH = (EVAL_SETS_DIR / f"{_name}.csv") if _name else None

def _load_pairs_csv(path: Path) -> list[tuple[str, str, int]]:
    pairs_df = pd.read_csv(path)
    needed = {"start", "target", "bfs_optimal"}
    if not needed.issubset(pairs_df.columns):
        raise ValueError(f"CSV must have columns {needed}, got {list(pairs_df.columns)}")
    if "graph_preset" in pairs_df.columns:
        bad = pairs_df["graph_preset"] != GRAPH_PRESET
        if bad.any():
            raise ValueError(
                f"CSV preset mismatch: {path.name} was saved for graph_preset={set(pairs_df.loc[bad, 'graph_preset'])} "
                f"but GRAPH_PRESET={GRAPH_PRESET!r}. This file is tied to a different island/word length. "
                f"Fix: use EVAL_SET_NAME=None in config so the name follows GRAPH_PRESET, "
                f"or set EVAL_SET_NAME to a CSV saved for this preset, "
                f"or FORCE_RESAMPLE_EVAL=True to resample and overwrite."
            )
    return [(str(r["start"]), str(r["target"]), int(r["bfs_optimal"])) for _, r in pairs_df.iterrows()]


def _sample_pairs() -> list[tuple[str, str, int]]:
    random.seed(RANDOM_SEED)
    out = []
    seen = set()
    attempts = 0
    max_attempts = N_PAIRS * 500
    while len(out) < N_PAIRS and attempts < max_attempts:
        attempts += 1
        a, b = random.sample(playable, 2)
        if (a, b) in seen:
            continue
        seen.add((a, b))
        try:
            d = nx.shortest_path_length(G, a, b)
        except nx.NetworkXNoPath:
            continue
        if DIST_MIN <= d <= DIST_MAX:
            out.append((a, b, d))
    if len(out) < N_PAIRS:
        print(f"Warning: only sampled {len(out)}/{N_PAIRS} pairs (relax DIST_* or increase attempts)")
    return out


def _save_eval_csv(path: Path, rows: list[tuple[str, str, int]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(rows, columns=["start", "target", "bfs_optimal"])
    df["graph_preset"] = GRAPH_PRESET
    df["random_seed"] = RANDOM_SEED
    df["dist_min"] = DIST_MIN
    df["dist_max"] = DIST_MAX
    df.to_csv(path, index=False)
    print(f"Saved eval set ({len(rows)} pairs) to {path}")


if LOAD_PAIRS_CSV:
    _p = Path(LOAD_PAIRS_CSV)
    eval_pairs = _load_pairs_csv(_p)
    print(f"Loaded {len(eval_pairs)} pairs from LOAD_PAIRS_CSV={_p}")
elif (
    EVAL_SET_PATH
    and EVAL_SET_PATH.exists()
    and EVAL_SET_REUSE_IF_EXISTS
    and not FORCE_RESAMPLE_EVAL
):
    eval_pairs = _load_pairs_csv(EVAL_SET_PATH)
    print(f"Loaded {len(eval_pairs)} pairs from eval set {EVAL_SET_PATH.name}")
else:
    eval_pairs = _sample_pairs()
    if SAVE_PAIRS_CSV:
        _save_eval_csv(Path(SAVE_PAIRS_CSV), eval_pairs)
    elif EVAL_SET_PATH:
        _save_eval_csv(EVAL_SET_PATH, eval_pairs)

print(f"Evaluation pairs: {len(eval_pairs)}")
print("Sample:", eval_pairs[:3])

Graph edges: 100%|██████████| 3863/3863 [00:00<00:00, 28018.75it/s]

Saved eval set (20 pairs) to C:\Users\doric\Documents\GitHub\word-ladder\data\eval_sets\test4croatian.csv
Evaluation pairs: 20
Sample: [('soda', 'duša', 3), ('besa', 'umre', 8), ('koga', 'kaki', 3)]


In [87]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model not found: {MODEL_PATH}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert = AutoModelForSequenceClassification.from_pretrained(str(MODEL_PATH))
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH))
bert = bert.to(device)
print(f"BERT on {device}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4390.84it/s]


BERT on cpu


In [90]:
# --- Your model (A* + BFS fallback) ---
bert_rows = []
t0 = time.time()
for start, target, opt in tqdm(eval_pairs, desc="BERT A*"):
    path, method = generate_path_astar(
        bert, tokenizer, vocab, start, target,
        device=device,
        max_expansions=MAX_EXPANSIONS,
        max_steps=MAX_STEPS,
    )
    steps = len(path) - 1 if path else None
    ok, _reason = (
        validate_ladder(path, vocab, WORD_LEN, require_in_graph_vocab=True)
        if path
        else (False, "no path")
    )
    reaches = bool(path and path[0] == start and path[-1] == target)
    success = ok and reaches
    bert_rows.append({
        "start": start,
        "target": target,
        "bfs_optimal": opt,
        "method": method,
        "steps": steps,
        "valid": success,
        "optimal_length": steps == opt if success and steps is not None else False,
    })

bert_time = time.time() - t0
df_bert = pd.DataFrame(bert_rows)
print(f"BERT total wall time: {bert_time:.1f}s ({bert_time/len(eval_pairs):.2f}s / pair)")
print(df_bert.groupby("method").size())
print("valid ladder:", df_bert["valid"].mean())
print("length optimal vs sampled BFS dist:", df_bert["optimal_length"].mean())
df_bert.head()

BERT A*: 100%|██████████| 20/20 [05:46<00:00, 17.33s/it]

BERT total wall time: 346.6s (17.33s / pair)
method
astar    20
dtype: int64
valid ladder: 1.0
length optimal vs sampled BFS dist: 1.0


,start,target,bfs_optimal,method,steps,valid,optimal_length
0,soda,duša,3,astar,3,True,True
1,besa,umre,8,astar,8,True,True
2,koga,kaki,3,astar,3,True,True
3,išta,gadi,7,astar,7,True,True
4,ulog,draž,8,astar,8,True,True


In [88]:
# --- GPT ---
req_graph = bool(globals().get("GPT_REQUIRE_GRAPH_VOCAB", True))
print(
    f"GPT_REQUIRE_GRAPH_VOCAB={req_graph} | "
    f"True => every word must be in island file; False => only length + one-letter steps"
)
if eval_pairs:
    _s, _t = eval_pairs[0][0], eval_pairs[0][1]
    _sys, _usr, _ = build_gpt_chat_payload(_s, _t, WORD_LEN)
    print(f"\n--- Prompt preview (first pair: {_s} -> {_t}) ---\n")
    print("[system]\n", _sys, "\n\n[user]\n", _usr, "\n")

gpt_rows = []
t0 = time.time()
for start, target, opt in tqdm(eval_pairs, desc="GPT"):
    path, raw, latency = gpt_word_ladder(start, target, WORD_LEN)
    if path:
        ok_loose, r_loose = validate_ladder(path, vocab, WORD_LEN, require_in_graph_vocab=False)
        ok_strict, r_strict = validate_ladder(path, vocab, WORD_LEN, require_in_graph_vocab=True)
        ok, reason = (ok_strict, r_strict) if req_graph else (ok_loose, r_loose)
    else:
        ok_loose = ok_strict = False
        ok, reason = False, "parse or empty"
        r_loose = r_strict = ""
    reaches = bool(ok and path and path[0] == start and path[-1] == target)
    on_graph = bool(
        path and ok_strict and path[0] == start and path[-1] == target
    )
    steps = len(path) - 1 if reaches else None
    frac = fraction_words_in_graph(path, vocab) if path else 0.0
    gpt_rows.append({
        "start": start,
        "target": target,
        "bfs_optimal": opt,
        "valid": reaches,
        "valid_on_graph": on_graph,
        "steps": steps,
        "optimal_length": steps == opt if reaches and steps is not None else False,
        "latency_sec": latency,
        "frac_words_in_graph": frac,
        "fail_reason": "" if reaches else (reason if path else "no parse"),
    })

gpt_time = time.time() - t0
df_gpt = pd.DataFrame(gpt_rows)
print(f"GPT total wall time: {gpt_time:.1f}s ({gpt_time/len(eval_pairs):.2f}s / pair)")
print("reaches target + valid (per mode):", df_gpt["valid"].mean())
print("... also valid on island graph:", df_gpt["valid_on_graph"].mean())
print("mean fraction of path words in graph (when any path returned):", df_gpt["frac_words_in_graph"].mean())
print("length optimal (when valid):", df_gpt.loc[df_gpt["valid"], "optimal_length"].mean())
print("mean API latency (per pair):", df_gpt["latency_sec"].mean())
df_gpt.head(10)

GPT_REQUIRE_GRAPH_VOCAB=False | True => every word must be in island file; False => only length + one-letter steps

--- Prompt preview (first pair: soda -> duša) ---

[system]
 You solve word ladders. Reply with ONLY a JSON array of lowercase strings: one word per step from start to target, no markdown, no commentary outside the array. 

[user]
 Language: Croatian (letters a-z plus č, ć, đ, š, ž).
Each word must have exactly 4 characters.
Each consecutive pair must differ in exactly ONE character position (same length).

Tiny example (4-letter, lowercase). First step in full: heat→head — change position 4 from t to d. Then only the deltas: head→heal (4:d→l); heal→hell (3:a→l). JSON: ["heat","head","heal","hell"]

--- Your task ---
Start: "soda"
Target: "duša"

Output ONLY a JSON array from start to target, e.g. ["soda", "...", "duša"] 



GPT: 100%|██████████| 20/20 [00:14<00:00,  1.41it/s]

GPT total wall time: 14.1s (0.71s / pair)
reaches target + valid (per mode): 0.1
... also valid on island graph: 0.0
mean fraction of path words in graph (when any path returned): 0.7633333333333333
length optimal (when valid): 0.0
mean API latency (per pair): 0.706648588180542


,start,target,bfs_optimal,valid,valid_on_graph,steps,optimal_length,latency_sec,frac_words_in_graph,fail_reason
0,soda,duša,3,False,False,NaN,False,1.132397,1.000000,not one-letter step: 'suda' -> 'duša'
1,besa,umre,8,False,False,NaN,False,0.495131,0.750000,not one-letter step: 'mesa' -> 'mrea'
2,koga,kaki,3,False,False,NaN,False,0.664749,0.666667,not one-letter step: 'koga' -> 'kaka'
3,išta,gadi,7,False,False,NaN,False,0.620391,0.666667,not one-letter step: 'gšta' -> 'gadi'
4,ulog,draž,8,False,False,NaN,False,0.661862,1.000000,not one-letter step: 'ulog' -> 'ulog'
5,takt,umor,8,False,False,NaN,False,0.735022,0.500000,not one-letter step: 'takt' -> 'trok'
6,šiva,prsi,4,False,False,NaN,False,0.918018,1.000000,not one-letter step: 'siva' -> 'siva'
7,diše,ruda,4,False,False,NaN,False,0.798158,0.800000,not one-letter step: 'ruse' -> 'ruda'
8,nađi,biro,5,False,False,NaN,False,0.741994,0.800000,not one-letter step: 'bari' -> 'biro'
9,bina,dlan,7,False,False,NaN,False,0.601827,1.000000,not one-letter step: 'dina' -> 'dlan'


In [89]:
# --- Gemini (same eval pairs & validation as GPT) ---
if gemini_client is None:
    print("Skipping Gemini (no client — set GEMINI_MODEL + GEMINI_API_KEY, run imports cell).")
    df_gemini = None
    gemini_time = None
else:
    req_graph = bool(globals().get("GPT_REQUIRE_GRAPH_VOCAB", True))
    print(
        f"GEMINI_MODEL={GEMINI_MODEL!r} | GPT_REQUIRE_GRAPH_VOCAB={req_graph} "
        "(same meaning as GPT eval)"
    )
    if eval_pairs:
        _s, _t = eval_pairs[0][0], eval_pairs[0][1]
        _sys, _usr, _ = build_gpt_chat_payload(_s, _t, WORD_LEN)
        print(f"\n--- Gemini prompt preview (first pair: {_s} -> {_t}) ---\n")
        print("[system]\n", _sys, "\n\n[user]\n", _usr, "\n")

    gemini_rows = []
    t0 = time.time()
    for start, target, opt in tqdm(eval_pairs, desc="Gemini"):
        path, raw, latency = gemini_word_ladder(start, target, WORD_LEN)
        if path:
            ok_loose, r_loose = validate_ladder(path, vocab, WORD_LEN, require_in_graph_vocab=False)
            ok_strict, r_strict = validate_ladder(path, vocab, WORD_LEN, require_in_graph_vocab=True)
            ok, reason = (ok_strict, r_strict) if req_graph else (ok_loose, r_loose)
        else:
            ok_loose = ok_strict = False
            ok, reason = False, "parse or empty"
            r_loose = r_strict = ""
        reaches = bool(ok and path and path[0] == start and path[-1] == target)
        on_graph = bool(path and ok_strict and path[0] == start and path[-1] == target)
        steps = len(path) - 1 if reaches else None
        frac = fraction_words_in_graph(path, vocab) if path else 0.0
        gemini_rows.append({
            "start": start,
            "target": target,
            "bfs_optimal": opt,
            "valid": reaches,
            "valid_on_graph": on_graph,
            "steps": steps,
            "optimal_length": steps == opt if reaches and steps is not None else False,
            "latency_sec": latency,
            "frac_words_in_graph": frac,
            "fail_reason": "" if reaches else (reason if path else "no parse"),
        })

    gemini_time = time.time() - t0
    df_gemini = pd.DataFrame(gemini_rows)
    print(f"Gemini total wall time: {gemini_time:.1f}s ({gemini_time/len(eval_pairs):.2f}s / pair)")
    print("reaches target + valid (per mode):", df_gemini["valid"].mean())
    print("... also valid on island graph:", df_gemini["valid_on_graph"].mean())
    print("mean fraction of path words in graph (when any path returned):", df_gemini["frac_words_in_graph"].mean())
    print("length optimal (when valid):", df_gemini.loc[df_gemini["valid"], "optimal_length"].mean())
    print("mean API latency (per pair):", df_gemini["latency_sec"].mean())
    df_gemini.head(10)


GEMINI_MODEL='gemini-2.5-flash' | GPT_REQUIRE_GRAPH_VOCAB=False (same meaning as GPT eval)

--- Gemini prompt preview (first pair: soda -> duša) ---

[system]
 You solve word ladders. Reply with ONLY a JSON array of lowercase strings: one word per step from start to target, no markdown, no commentary outside the array. 

[user]
 Language: Croatian (letters a-z plus č, ć, đ, š, ž).
Each word must have exactly 4 characters.
Each consecutive pair must differ in exactly ONE character position (same length).

Tiny example (4-letter, lowercase). First step in full: heat→head — change position 4 from t to d. Then only the deltas: head→heal (4:d→l); heal→hell (3:a→l). JSON: ["heat","head","heal","hell"]

--- Your task ---
Start: "soda"
Target: "duša"

Output ONLY a JSON array from start to target, e.g. ["soda", "...", "duša"] 



Gemini: 100%|██████████| 20/20 [00:55<00:00,  2.77s/it]

Gemini total wall time: 55.5s (2.77s / pair)
reaches target + valid (per mode): 0.15
... also valid on island graph: 0.05
mean fraction of path words in graph (when any path returned): 0.2683333333333333
length optimal (when valid): 0.3333333333333333
mean API latency (per pair): 2.773458790779114


In [91]:
# --- Side-by-side summary ---
df_gemini = globals().get("df_gemini")
gemini_time = globals().get("gemini_time")
def _mean_steps_str(df):
    v = df.loc[df["valid"], "steps"].mean()
    return f"{v:.2f}" if pd.notna(v) else "n/a"

_cols = {
    "metric": [
        "valid / success",
        "optimal length (vs sampled opt)",
        "mean steps (when success)",
        "total wall seconds",
    ],
    "BERT_Astar_BFS": [
        f"{df_bert['valid'].mean():.1%}",
        f"{df_bert['optimal_length'].mean():.1%}",
        f"{df_bert.loc[df_bert['valid'], 'steps'].mean():.2f}",
        f"{bert_time:.1f}",
    ],
    "GPT": [
        f"{df_gpt['valid'].mean():.1%}",
        f"{df_gpt['optimal_length'].mean():.1%}",
        _mean_steps_str(df_gpt),
        f"{gpt_time:.1f}",
    ],
}
if df_gemini is not None:
    _cols["Gemini"] = [
        f"{df_gemini['valid'].mean():.1%}",
        f"{df_gemini['optimal_length'].mean():.1%}",
        _mean_steps_str(df_gemini),
        f"{gemini_time:.1f}",
    ]
else:
    _cols["Gemini"] = ["—", "—", "—", "—"]

summary = pd.DataFrame(_cols)
_models = f"GPT={GPT_MODEL}"
if df_gemini is not None:
    _models += f" | Gemini={GEMINI_MODEL}"
print(f"Preset={GRAPH_PRESET} | pairs={len(eval_pairs)} | {_models}")
print(summary.to_string(index=False))

bad = df_gpt.loc[~df_gpt["valid"], ["start", "target", "bfs_optimal", "fail_reason"]]
if len(bad):
    print("\nGPT failures (sample):")
    print(bad.head(15).to_string())
if df_gemini is not None:
    bad_g = df_gemini.loc[~df_gemini["valid"], ["start", "target", "bfs_optimal", "fail_reason"]]
    if len(bad_g):
        print("\nGemini failures (sample):")
        print(bad_g.head(15).to_string())

Preset=croatian_4 | pairs=20 | GPT=gpt-5.4-mini | Gemini=gemini-2.5-flash
                         metric BERT_Astar_BFS   GPT Gemini
                valid / success         100.0% 10.0%  15.0%
optimal length (vs sampled opt)         100.0%  0.0%   5.0%
      mean steps (when success)           5.30  3.50   3.33
             total wall seconds          346.6  14.1   55.5

GPT failures (sample):
   start target  bfs_optimal                            fail_reason
0   soda   duša            3  not one-letter step: 'suda' -> 'duša'
1   besa   umre            8  not one-letter step: 'mesa' -> 'mrea'
2   koga   kaki            3  not one-letter step: 'koga' -> 'kaka'
3   išta   gadi            7  not one-letter step: 'gšta' -> 'gadi'
4   ulog   draž            8  not one-letter step: 'ulog' -> 'ulog'
5   takt   umor            8  not one-letter step: 'takt' -> 'trok'
6   šiva   prsi            4  not one-letter step: 'siva' -> 'siva'
7   diše   ruda            4  not one-letter step: 'ruse' 